# Medley PD — 통합 실행 노트북 (Colab)

**런타임 → 런타임 유형 변경 → GPU** 후 위에서부터 실행하세요.

| 단계 | 명령 | 바꾸는 것 |
|---|---|---|
| 데이터 받기 | `run.py download --dataset D` | 데이터셋 |
| 구조 확인 | `run.py inspect --dataset D` | 데이터셋 |
| 피처 추출 | `run.py extract --dataset D --encoder E` | 데이터셋 × 인코더 |
| 기관 내 평가 | `run.py cv --dataset D --encoder E --task T` | 데이터셋 × 인코더 × 과제 |
| 기관 간 평가 | `run.py cross --train D1 --test D2 --encoder E --task T` | |

공통 전처리·평가 설정은 `configs/base.yaml` 한 곳에만 있습니다.

## 0. 설치

In [ ]:
!pip install -q mne openneuro-py einops huggingface_hub pyyaml timm safetensors
import torch; print('GPU:', torch.cuda.is_available())

## 1. Drive 연결 + 코드 준비
GitHub 에서 받거나(A), Drive 에 올린 코드 폴더를 복사합니다(B). 마지막 `ls` 에 `run.py configs medley` 가 보여야 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/medley_pd'        # 피처·결과·split 이 저장될 Drive 폴더
CODE = '/content/medley_pd'
os.makedirs(WORK, exist_ok=True)

%cd /content
!rm -rf {CODE}
# (A) GitHub 에서 받기 — 저장소 안 폴더 위치에 맞게 경로 확인
!git clone -q https://github.com/jeongwon8u8/Medley_skkai /content/repo && cp -r /content/repo/medley_pd {CODE}
# (B) Drive 에 올린 코드 폴더를 쓸 때는 위 줄 대신:
# !cp -r /content/drive/MyDrive/medley_pd/medley_pd {CODE}
%cd {CODE}
!ls

## 2. CBraMod 원본 코드 + 가중치
기존 노트북에서 받아 둔 것이 있으면 그대로 씁니다.

In [ ]:
import shutil
CBRAMOD_REPO, WEIGHTS = None, None
for base in [WORK, '/content/drive/MyDrive/pd_eeg_project', '/content/drive/MyDrive/mumtaz_hcmdd_project', '/content/drive/MyDrive/tdbrain_hcmdd_project']:
    if os.path.exists(f'{base}/CBraMod/models/cbramod.py') and os.path.exists(f'{base}/pretrained_weights.pth'):
        CBRAMOD_REPO, WEIGHTS = f'{base}/CBraMod', f'{base}/pretrained_weights.pth'; break
if CBRAMOD_REPO is None:
    from huggingface_hub import hf_hub_download
    CBRAMOD_REPO, WEIGHTS = f'{WORK}/CBraMod', f'{WORK}/pretrained_weights.pth'
    !git clone -q https://github.com/wjq-learning/CBraMod "$CBRAMOD_REPO"
    shutil.copy(hf_hub_download('weighting666/CBraMod', 'pretrained_weights.pth'), WEIGHTS)
print(CBRAMOD_REPO, WEIGHTS)

## 3. 공통 경로 옵션
데이터는 빠른 읽기를 위해 로컬(`/content/data`)에 두고, 피처·결과·split 은 Drive 에 저장합니다.

In [ ]:
DATA = '/content/data'
S = (f'--set paths.data_root={DATA} --set paths.feature_root={WORK}/features '
     f'--set paths.result_root={WORK}/results --set paths.split_root={WORK}/splits')
CB = f'--set encoder.repo={CBRAMOD_REPO} --set encoder.weights={WEIGHTS}'      # CBraMod 전용
os.makedirs(DATA, exist_ok=True)

## 4. PD-EEG
기존 노트북에서 Drive 에 받아 둔 `ds007526` 이 있으면 복사만 하고, **기존 CBraMod 피처 캐시도 그대로 재사용**합니다(같은 전처리·같은 가중치).

In [ ]:
OLD = '/content/drive/MyDrive/pd_eeg_project'
if os.path.isdir(f'{OLD}/ds007526'):
    shutil.copytree(f'{OLD}/ds007526', f'{DATA}/ds007526', dirs_exist_ok=True)
else:
    !python run.py download --dataset pd_eeg $S

In [ ]:
!python run.py inspect --dataset pd_eeg $S

In [ ]:
# 기존 캐시(<sid>_EO.npz)가 있으면 복사만 함 → 거의 바로 끝남
LEGACY = f'{OLD}/out/features'
!python run.py extract --dataset pd_eeg --encoder cbramod $S $CB --set paths.legacy_feature_dir={LEGACY}

In [ ]:
# 실험 ① — MoCA. C0/C1/C2 × (clinical-only, EEG-only, stack, concat) + mean·noise baseline, seed 10개
!python run.py cv --dataset pd_eeg --encoder cbramod --task moca $S

In [ ]:
# 실험 ② — CTT (⚠ ctt 열 정의 확인 후). C1 = ⑤-A, C1_moca = ⑤-B
!python run.py cv --dataset pd_eeg --encoder cbramod --task ctt $S

## 5. Iowa Rest
**`inspect` 결과를 보고 `configs/datasets/iowa_rest.yaml` 의 `columns`, `file_pattern` 을 먼저 고치세요.** (열 이름·group 값·파일 이름 확인)

In [ ]:
!python run.py download --dataset iowa_rest $S
!python run.py inspect --dataset iowa_rest $S

In [ ]:
!python run.py extract --dataset iowa_rest --encoder cbramod $S $CB --limit 3     # 시험 실행

In [ ]:
!python run.py extract --dataset iowa_rest --encoder cbramod $S $CB
!python run.py cv --dataset iowa_rest --encoder cbramod --task moca $S

## 6. 기관 간 검증 (공통 변수 C0 만)

In [ ]:
!python run.py cross --train pd_eeg --test iowa_rest --encoder cbramod --task moca $S
!python run.py cross --train iowa_rest --test pd_eeg --encoder cbramod --task moca $S

## 7. S-CEReBrO
BioFoundation PR #18 (commit `361c017`) 코드와 Hugging Face `PulpBio/S-CEReBrO` 가중치를 받습니다.
전처리(200 Hz, 0.3–75 Hz, 노치, 5초)는 CBraMod 와 같고, 인코더 안에서 채널별 min-max [-1, 1] 정규화와 전극 좌표(기준전극 포함)가 추가됩니다.
**Iowa 는 `iowa_rest.yaml` 의 `reference`(기록 기준전극)를 채워야 실행됩니다.**

In [ ]:
BF = f'{WORK}/BioFoundation'
if not os.path.exists(f'{BF}/models/s_cerebro.py'):
    !git clone -q https://github.com/pulp-bio/BioFoundation "$BF"
    !cd "$BF" && git fetch -q origin 361c017ce0018141391d068e85fd6352fa32a9ef && git checkout -q FETCH_HEAD
from huggingface_hub import hf_hub_download
SCW = hf_hub_download('PulpBio/S-CEReBrO', 'S_CEReBrO.safetensors', local_dir=f'{WORK}/S-CEReBrO')
SC = f'--set encoder.repo={BF} --set encoder.weights={SCW}'
print(SC)

In [ ]:
!python run.py extract --dataset pd_eeg --encoder scerebro $S $SC --limit 3     # 시험 실행: loaded=... 확인
!python run.py extract --dataset pd_eeg --encoder scerebro $S $SC
!python run.py cv --dataset pd_eeg --encoder scerebro --task moca $S
# Iowa: reference 채운 뒤
# !python run.py extract --dataset iowa_rest --encoder scerebro $S $SC
# !python run.py cv --dataset iowa_rest --encoder scerebro --task moca $S
# !python run.py cross --train pd_eeg --test iowa_rest --encoder scerebro --task moca $S

## 8. 결과 모아 보기

In [ ]:
import pandas as pd, glob
pd.set_option('display.width', 220)
for f in sorted(glob.glob(f'{WORK}/results/**/summary.csv', recursive=True)):
    print('\n###', f.replace(WORK + '/results/', ''))
    s = pd.read_csv(f)
    print(s[['model', 'avg_n', 'avg_MAE', 'MAE_lo', 'MAE_hi', 'avg_Spearman']].round(3).to_string(index=False))